***EDA Comunas***

Dataset necesario para graficar las comunas.

En total hay 15 comunas.

No hay duplicados.

Descarto el campo `_rescued_data`, ya que vino totalmente nulo.

No hay valores raros, como "





In [0]:
%sql
describe proyecto_final.raw.comunas_bronze

In [0]:

%sql
select * from proyecto_final.raw.comunas_bronze

In [0]:
%sql

-- Nulos
select 
count(*) as total_registros,
count(*) - count(id) as id_nulos,
count(*) - count(objeto) as objeto_nulos,
count(*) - count(comuna) as comuna_nulos,
count(*) - count(barrios) as barrios_nulos,
count(*) - count(perimetro) as perimetro_nulos,
count(*) - count(area) as area_nulos,
count(*) - count(geometry) as geometry_nulos,
count(*) - count(_rescued_data) as _rescued_data_nulos
from proyecto_final.raw.comunas_bronze

-- No hay nulos

In [0]:
%sql
--Duplicados
select comuna,
count(*) as cantidad
from proyecto_final.raw.comunas_bronze
group by comuna
having count(*) > 1
order by cantidad desc


In [0]:
%sql
-- Busco si en los puntos de geometry hay alguno invalido
SELECT 
    comuna,
    geometry,
    st_isvalid(st_geomfromwkt(geometry)) as es_valida
FROM proyecto_final.raw.barrios_bronze
WHERE st_isvalid(st_geomfromwkt(geometry)) = false;

In [0]:
%sql
--Comunas con mayor area numerados
select 
row_number() over (order by area desc) as pos,
comuna,
barrios,
area
from proyecto_final.raw.comunas_bronze
limit 5;


In [0]:
%sql

CREATE OR REPLACE VIEW v_comunas_limpieza AS
SELECT 
  id AS comuna_id,
  LOWER(TRIM(objeto)) AS tipo_entidad,
  comuna AS comuna,
  lower(TRIM(barrios)) AS barrios,
  ROUND(perimetro, 2) AS perimetro_m,
  ROUND(area, 2) AS area_m2,
  geometry AS coordenadas
FROM proyecto_final.raw.comunas_bronze
WHERE comuna IS NOT NULL 
  AND geometry IS NOT NULL;

select * from v_comunas_limpieza;